## 02. Data Cleaning & Preparation

### 1. Data Loading & Setup

In [1]:
# Import libraries and configure dataset paths

import pandas as pd
import csv
import re
from pathlib import Path
from collections import Counter

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

csv.field_size_limit(2**31 - 1)

files = {
    "Movies": "movie_data.csv",
    "Ratings": "ratings_export.csv",
    "Users": "users_export.csv"
}

for name, filename in files.items():
    file_path = raw_path / filename
    print(f"{name}: {'Found' if file_path.is_file() else 'Missing'}")


Movies: Found
Ratings: Found
Users: Found


### 2. Movies Data Cleaning

#### 2.1 CSV Structure Repair

In [2]:
# Repair fragmented movie records

source_file = raw_path / "movie_data.csv"
output_file = processed_path / "movie_data_repaired.csv"

repair_counts = Counter()
pending_row = None

with open(source_file, encoding="utf-8-sig", newline="") as source:
    with open(output_file, "w", encoding="utf-8", newline="") as output:

        reader = csv.reader(source)
        writer = csv.writer(output)

        header = next(reader)
        writer.writerow(header)

        for row in reader:

            if pending_row is None:

                if len(row) == 19:
                    writer.writerow(row)
                    repair_counts["Valid records"] += 1

                elif len(row) == 9:
                    pending_row = row

                else:
                    raise ValueError(f"Unexpected row length: {len(row)}")

            else:

                if len(row) == 1:
                    pending_row[-1] += "\n" + row[0]

                elif len(row) == 11:
                    pending_row[-1] += "\n" + row[0]

                    repaired_row = pending_row + row[1:]

                    if len(repaired_row) != 19:
                        raise ValueError("Repair failed: incorrect column count")

                    writer.writerow(repaired_row)
                    repair_counts["Repaired records"] += 1

                    pending_row = None

                else:
                    raise ValueError(f"Unexpected continuation: {len(row)}")

        if pending_row is not None:
            raise ValueError("Incomplete movie record at end of file")

print(repair_counts)
print("Total movie records:", sum(repair_counts.values()))


Counter({'Valid records': 285390, 'Repaired records': 114})
Total movie records: 285504


In [3]:
# Validate repaired CSV structure

total_records = 0
invalid_rows = 0
invalid_ids = 0
duplicate_ids = 0

seen_ids = set()

with open(output_file, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

    id_index = header.index("_id")

    for row in reader:
        total_records += 1

        if len(row) != len(header):
            invalid_rows += 1
            continue

        movie_id = row[id_index]

        if not re.fullmatch(r"[0-9a-fA-F]{24}", movie_id):
            invalid_ids += 1

        if movie_id in seen_ids:
            duplicate_ids += 1

        seen_ids.add(movie_id)

print("Total records:", total_records)
print("Invalid row structures:", invalid_rows)
print("Invalid _id values:", invalid_ids)
print("Duplicate _id values:", duplicate_ids)


Total records: 285504
Invalid row structures: 0
Invalid _id values: 0
Duplicate _id values: 0


In [4]:
# Inspect recovered movie records

repaired_ids = set()

with open(source_file, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    next(reader)

    for row in reader:
        if len(row) == 9:
            repaired_ids.add(row[0])

movies_repaired = pd.read_csv(
    output_file,
    dtype=str,
    keep_default_na=False
)

recovered_movies = movies_repaired[
    movies_repaired["_id"].isin(repaired_ids)
]

print("Expected recovered records:", len(repaired_ids))
print("Recovered records found:", len(recovered_movies))


Expected recovered records: 114
Recovered records found: 114


In [5]:
# Validate numeric fields in recovered records

numeric_columns = [
    "popularity",
    "runtime",
    "vote_average",
    "vote_count",
    "year_released"
]

for col in numeric_columns:
    values = recovered_movies[col].replace(
        ["", "null", "none", "nan"], pd.NA
    ).dropna()

    invalid_count = pd.to_numeric(
        values, errors="coerce"
    ).isna().sum()

    print(f"{col}: {invalid_count} invalid values")


popularity: 0 invalid values
runtime: 0 invalid values
vote_average: 0 invalid values
vote_count: 0 invalid values
year_released: 0 invalid values


#### 2.2 Missing Values


In [6]:
# Missing values summary

missing_mask = movies_repaired.apply(
    lambda col: col.isna() |
    col.astype("string").str.strip().str.lower().isin(
        ["", "null", "none", "nan"]
    )
)


missing_counts = missing_mask.sum()

missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": (missing_counts / len(movies_repaired) * 100).round(2)
}).sort_values("Missing Count", ascending=False)

display(missing_summary[missing_summary["Missing Count"] > 0])


,Missing Count,Missing %
imdb_id,41702,14.61
imdb_link,41702,14.61
overview,24284,8.51
image_url,21097,7.39
runtime,15449,5.41
release_date,14454,5.06
genres,10632,3.72
vote_count,10632,3.72
production_countries,10632,3.72
popularity,10632,3.72


In [7]:
# Missing values patterns and genre coverage

metadata_columns = [
    "genres", "popularity", "vote_average",
    "vote_count", "original_language",
    "production_countries", "spoken_languages"
]

metadata_missing = missing_mask[metadata_columns]
affected_movies = movies_repaired.loc[
    metadata_missing.all(axis=1)
]

print("All metadata fields missing:",
      metadata_missing.all(axis=1).sum())
print("Any metadata field missing:",
      metadata_missing.any(axis=1).sum())

# Check other fields in affected movies
other_columns = [
    "overview", "runtime", "release_date",
    "year_released", "movie_title", "movie_id"
]

print("\nOther missing fields in affected movies:")
print(missing_mask.loc[affected_movies.index, other_columns].sum())

# Check genre coverage
genres = movies_repaired["genres"].str.strip().str.lower()

missing_genres = missing_mask["genres"].sum()
empty_genres = (genres == "[]").sum()

print("\nMissing genres:", missing_genres)
print("Empty genre lists:", empty_genres)
print("Movies without genres:", missing_genres + empty_genres)

# Check linked ratings
affected_ids = set(affected_movies["movie_id"])
matched_ratings = 0
rated_movies = set()

for chunk in pd.read_csv(
    raw_path / "ratings_export.csv",
    usecols=["movie_id"],
    chunksize=100_000
):
    matches = chunk[chunk["movie_id"].isin(affected_ids)]
    matched_ratings += len(matches)
    rated_movies.update(matches["movie_id"])

print("\nAffected movies with ratings:", len(rated_movies))
print("Linked ratings:", matched_ratings)


All metadata fields missing: 10632
Any metadata field missing: 10632

Other missing fields in affected movies:
overview         10632
runtime          10632
release_date     10632
year_released     3261
movie_title       2165
movie_id             0
dtype: int64

Missing genres: 10632
Empty genre lists: 52346
Movies without genres: 62978

Affected movies with ratings: 10632
Linked ratings: 167900


In [8]:
# Validate and recover missing release years

existing_year = pd.to_numeric(
    movies_repaired["year_released"],
    errors="coerce"
)

release_dates = pd.to_datetime(
    movies_repaired["release_date"],
    errors="coerce"
)

year_from_date = release_dates.dt.year

# Check year differences
both_available = existing_year.notna() & year_from_date.notna()
mismatches = both_available & (existing_year != year_from_date)

year_difference = existing_year - year_from_date

print("Year mismatches:", mismatches.sum())
print("Large differences (5+ years):",
      (year_difference.abs() >= 5).sum())

# Recover missing years
recoverable = existing_year.isna() & year_from_date.notna()

movies_repaired["year_released_clean"] = (
    existing_year.fillna(year_from_date).astype("Int64")
)

# Track year source
movies_repaired["year_source"] = "original"
movies_repaired.loc[recoverable, "year_source"] = "release_date"
movies_repaired.loc[
    movies_repaired["year_released_clean"].isna(),
    "year_source"
] = "missing"

print("\nYear source:")
print(movies_repaired["year_source"].value_counts())

# Check December 31 pattern
dec31_count = movies_repaired.loc[
    recoverable, "release_date"
].str.endswith("-12-31").sum()

print("\nRecovered dates ending in December 31:", dec31_count)


Year mismatches: 5007
Large differences (5+ years): 109

Year source:
year_source
original        277932
missing           7042
release_date       530
Name: count, dtype: int64

Recovered dates ending in December 31: 476


In [9]:
# Standardize missing values and validate results

text_columns = movies_repaired.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    movies_repaired[col] = movies_repaired[col].replace(
        r"(?i)^\s*(?:null|none|nan)?\s*$",
        pd.NA,
        regex=True
    )

print("Total movie records:", len(movies_repaired))

print("\nMissing values after standardization:")
print(
    movies_repaired.isna().sum()
    .sort_values(ascending=False)
    .loc[lambda counts: counts > 0]
)

print("\nRelease year sources:")
print(movies_repaired["year_source"].value_counts())


Total movie records: 285504

Missing values after standardization:
imdb_link               41702
imdb_id                 41702
overview                24284
image_url               21097
runtime                 15449
release_date            14454
genres                  10632
vote_count              10632
original_language       10632
production_countries    10632
popularity              10632
vote_average            10632
spoken_languages        10632
year_released            7572
year_released_clean      7042
tmdb_link                5587
tmdb_id                  5587
movie_title              2167
movie_id                    2
dtype: int64

Release year sources:
year_source
original        277932
missing           7042
release_date       530
Name: count, dtype: int64
